In [45]:
import os
import pandas as pd

from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3.common.logger import configure
from finrl import config_tickers
from finrl.main import check_and_make_directories
from finrl.config import INDICATORS, TRAINED_MODEL_DIR, RESULTS_DIR

check_and_make_directories([TRAINED_MODEL_DIR])

In [46]:
"""
Trade-Republic-Kosten für das FinRL StockTradingEnv (Stand: September 2026).

Berücksichtigt nur:
  - Ordergebühr: 1 € (Bestpreis) bzw. 2 € (Direktpreis) fix pro Order
  - Transaktionskosten: geschätzter Spread in Prozent
"""
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv


class TradeRepublicStockTradingEnv(StockTradingEnv):
    def __init__(self, df, *, order_fee=1.0, spread_pct=0.0005, **kwargs):
        self.order_fee = order_fee      # 1.0 = Bestpreis, 2.0 = Direktpreis
        self.spread_pct = spread_pct    # z. B. 0.0005 = 0,05 %

        # Prozentuale Kosten des Basis-Envs abschalten
        n = kwargs["stock_dim"]
        kwargs["buy_cost_pct"] = [0.0] * n
        kwargs["sell_cost_pct"] = [0.0] * n
        super().__init__(df=df, **kwargs)

    def _turbulent(self):
        return (self.turbulence_threshold is not None
                and self.turbulence >= self.turbulence_threshold)

    def _sell_stock(self, index, action):
        price = self.state[1 + index]
        held = self.state[1 + self.stock_dim + index]
        if price <= 0 or held <= 0:
            return 0

        # Bei Turbulenz alles verkaufen (wie im Basis-Env)
        shares = int(held if self._turbulent() else min(abs(action), held))
        proceeds = price * shares * (1 - self.spread_pct)
        if shares <= 0 or proceeds <= self.order_fee:
            return 0

        self.state[0] += proceeds - self.order_fee
        self.state[1 + self.stock_dim + index] -= shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

    def _buy_stock(self, index, action):
        price = self.state[1 + index]
        if price <= 0 or self._turbulent():
            return 0

        ask = price * (1 + self.spread_pct)
        affordable = int((self.state[0] - self.order_fee) // ask)
        shares = int(min(affordable, action))
        if shares <= 0:
            return 0

        self.state[0] -= ask * shares + self.order_fee
        self.state[1 + self.stock_dim + index] += shares
        self.cost += price * shares * self.spread_pct + self.order_fee
        self.trades += 1
        return shares

In [47]:
#Load Data from 01_data_ops_notebook
train = pd.read_csv('../data/train.csv')
date = train.columns[0]
train.index = pd.factorize(train[date])[0]
train.index.names = ['']
train.head()


,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
,,,,,,,,,,,,,,,,,,
0,2009-01-02,A,10.259192,10.297096,9.798034,9.854888,4236220.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,10.259192,10.259192,39.189999,0.0
0,2009-01-02,AAPL,2.714298,2.722973,2.547104,2.568639,746015200.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,2.714298,2.714298,39.189999,0.0
0,2009-01-02,ABT,17.199024,17.292149,16.816896,17.189391,13163193.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,17.199024,17.199024,39.189999,0.0
0,2009-01-02,ACGL,7.287052,7.395878,7.226829,7.395878,3445200.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,7.287052,7.287052,39.189999,0.0
0,2009-01-02,ACN,24.225479,24.319015,23.412449,23.412449,3787400.0,4.0,0.0,10.92428,9.94155,100.0,66.666667,100.0,24.225479,24.225479,39.189999,0.0


In [48]:
#Setup Environment
stock_dimensions = len(train.tic.unique())
state_space = 1 + 2 * stock_dimensions + len(INDICATORS)*stock_dimensions
print(f"Stock Dimensions:  {stock_dimensions}, State Space: {state_space}")

Stock Dimensions:  414, State Space: 4141


In [49]:
buy_cost_list = sell_cost_list = [0] * stock_dimensions
num_stock_shares = [0] * stock_dimensions

env_tweeks = {
    "hmax": 100,
    "initial_amount": 1000,
    "num_stock_shares": num_stock_shares,
    "buy_cost_pct": buy_cost_list,
    "sell_cost_pct": sell_cost_list,
    "state_space": state_space,
    "stock_dim": stock_dimensions,
    "tech_indicator_list": INDICATORS,
    "action_space": stock_dimensions,
    "reward_scaling": 1e-4
}
#Setup of the environment with given attributes and data set
e_training_gym = TradeRepublicStockTradingEnv(df = train, **env_tweeks)


In [50]:
#Environment for Training

env_train, _ = e_training_gym.get_sb_env()
print(type(env_train))

<class 'stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv'>


In [51]:
#Setup Agent

agent = DRLAgent(env = env_train)

#Set if_using_MODEL = true if u want to use other algorithms
if_using_a2c = False
if_using_ddpg = False
if_using_ppo = True
if_using_td3 = False
if_using_sac = True

In [52]:
#Using model A2C
model_a2c =agent.get_model("a2c")

if if_using_a2c:
    #setup new logger
    tmp_path = RESULTS_DIR + '/a2c'
    new_logger_a2c = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_a2c.set_logger(new_logger_a2c)

#Using model DDPG
model_ddpg =agent.get_model("ddpg")

if if_using_ddpg:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ddpg'
    new_logger_ddpg = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ddpg.set_logger(new_logger_ddpg)

#Using model PPO
model_ppo =agent.get_model("ppo")

if if_using_ppo:
    #setup new logger
    tmp_path = RESULTS_DIR + '/ppo'
    new_logger_ppo = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_ppo.set_logger(new_logger_ppo)

#Using model TD3
##model_td3 =agent.get_model("td3")

#if if_using_td3:
    #setup new logger
    #tmp_path = RESULTS_DIR + '/td3'
    #new_logger_td3 = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
   # model_td3.set_logger(new_logger_td3)

#Using model SAC
model_sac =agent.get_model("sac")

if if_using_sac:
    #setup new logger
    tmp_path = RESULTS_DIR + '/sac'
    new_logger_sac = configure(tmp_path, ["stdout", "csv", "tensorboard"])
    #Set new logger
    model_sac.set_logger(new_logger_sac)

{'n_steps': 5, 'ent_coef': 0.01, 'learning_rate': 0.0007}
Using cpu device
{'batch_size': 128, 'buffer_size': 50000, 'learning_rate': 0.001}
Using cpu device
{'n_steps': 2048, 'ent_coef': 0.01, 'learning_rate': 0.00025, 'batch_size': 64}
Using cpu device
Logging to results/ppo
{'batch_size': 64, 'buffer_size': 100000, 'learning_rate': 0.0001, 'learning_starts': 100, 'ent_coef': 'auto_0.1'}
Using cpu device
Logging to results/sac


In [53]:
#Use loggs to train data. Add aditional blocks for other models if needed
if if_using_a2c:
    trained_a2c = agent.train_model(
        model = model_a2c,
        tb_log_name = 'a2c',
        total_timesteps = 500 #Increase for more learning
    )
if if_using_ddpg:
    trained_ddpg = agent.train_model(
        model = model_ddpg,
        tb_log_name = 'ddpg',
        total_timesteps = 500 #Increase for more learning
    )
if if_using_ppo:
    trained_ppo = agent.train_model(
        model = model_ppo,
        tb_log_name = 'ppo',
        total_timesteps = 500 #Increase for more learning
    )
if if_using_sac:
    trained_sac = model_sac.learn(
        total_timesteps = 500, #Increase for more learning
        tb_log_name = 'sac'
    )

---------------------------------------
| time/              |                |
|    fps             | 12             |
|    iterations      | 1              |
|    time_elapsed    | 163            |
|    total_timesteps | 2048           |
| train/             |                |
|    reward          | 0.0            |
|    reward_max      | 0.009439143    |
|    reward_mean     | -4.8775095e-05 |
|    reward_min      | -0.008554451   |
---------------------------------------


In [54]:
if if_using_a2c:
    trained_a2c.save(TRAINED_MODEL_DIR + "/agent_a2c")if if_using_a2c else None
if if_using_ddpg:
    trained_ddpg.save(TRAINED_MODEL_DIR + "/agent_ddpg")if if_using_ddpg else None
if if_using_ppo:
    trained_ppo.save(TRAINED_MODEL_DIR + "/agent_ppo")if if_using_ppo else None
if if_using_sac:
    trained_sac.save(TRAINED_MODEL_DIR + "/agent_sac")if if_using_sac else None


In [55]:
# Portfolio am Ende des Trainingszeitraums auswerten
if if_using_a2c:
    portfolio_model = trained_a2c
elif if_using_ddpg:
    portfolio_model = trained_ddpg
elif if_using_ppo:
    portfolio_model = trained_ppo
elif if_using_sac:
    portfolio_model = trained_sac
else:
    raise ValueError("Aktiviere mindestens ein trainiertes Modell.")

portfolio_env = TradeRepublicStockTradingEnv(df=train, **env_tweeks)
reset_result = portfolio_env.reset()
obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result

for _ in range(len(train.index.unique())):
    action, _ = portfolio_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = portfolio_env.step(action)
    if terminated or truncated:
        break

stock_count = portfolio_env.stock_dim
state = portfolio_env.state
cash = float(state[0])
prices = state[1:1 + stock_count]
shares = state[1 + stock_count:1 + 2 * stock_count]

ticker_order = train.tic.drop_duplicates().tolist()
portfolio = pd.DataFrame({
    "Aktie": ticker_order,
    "Stueck": shares,
    "Kurs_EUR": prices,
})
portfolio["Wert_EUR"] = portfolio["Stueck"] * portfolio["Kurs_EUR"]
total_value = cash + portfolio["Wert_EUR"].sum()
portfolio["Anteil_Prozent"] = portfolio["Wert_EUR"] / total_value * 100
portfolio = portfolio.loc[portfolio["Stueck"] > 0].sort_values("Wert_EUR", ascending=False)

print(f"Barbestand: {cash:.2f} EUR | Gesamtportfoliowert: {total_value:.2f} EUR")
if portfolio.empty:
    print("Am Ende des Zeitraums wurden keine Aktien gehalten.")
else:
    display(portfolio.reset_index(drop=True).round({"Kurs_EUR": 2, "Wert_EUR": 2, "Anteil_Prozent": 2}))

Barbestand: 0.10 EUR | Gesamtportfoliowert: 10249.03 EUR


,Aktie,Stueck,Kurs_EUR,Wert_EUR,Anteil_Prozent
0,INTU,11,331.03,3641.28,35.53
1,MCO,5,280.13,1400.66,13.67
2,TT,12,109.45,1313.39,12.81
3,ABT,10,97.87,978.65,9.55
4,DPZ,2,378.07,756.13,7.38
5,LLY,5,139.43,697.13,6.80
6,DRI,4,72.11,288.43,2.81
7,GIS,5,50.86,254.31,2.48
8,PSKY,10,24.15,241.52,2.36
9,NVDA,14,13.30,186.17,1.82
